# Build the \(K^0_S\)-constrained 2D TPC correction map

This notebook reads the `MakeK0s2DClusterCorrectionVotes_v3.C` output.

The final map is **not split in \(p_T\)**. For every detector bin, each available
charge-\(p_T\) vote histogram is first normalized to unit integral. The normalized
histograms are then averaged with equal category weight.

Separate \(p_T\) panels are retained only for the \(R1\), \(R2\), and \(R3\)
method illustrations.


In [1]:
from pathlib import Path
import math
import ROOT as root

%jsroot on

root.gStyle.SetOptStat(0)
root.gStyle.SetOptTitle(0)
root.gStyle.SetPalette(root.kBird)
root.gStyle.SetPadTickX(1)
root.gStyle.SetPadTickY(1)


Welcome to JupyROOT 6.30/06


In [2]:
# ============================================================
# Configuration
# ============================================================

selection_name = "cut03_baseline"  # or "cut07_pairDCA_5mm"
input_file = Path("input/map_k0s_pp_pion_qa_v3.root")
output_file = Path("output/k0s_2d_cluster_correction_map_exact_xy_cut03.root")
plot_dir = Path("output/k0s_2d_cluster_correction_plots")

plot_dir.mkdir(parents=True, exist_ok=True)

sides = ["side0", "side1"]
charges = ["qplus", "qminus"]

pt_bins = [
    "pt_0p2_0p4",
    "pt_0p4_0p7",
    "pt_0p7_1p2",
    "pt_1p2_1p8",
    "pt_1p8_5p0",
]

pt_labels = {
    "pt_0p2_0p4": "0.2 < p_{T} < 0.4",
    "pt_0p4_0p7": "0.4 < p_{T} < 0.7",
    "pt_0p7_1p2": "0.7 < p_{T} < 1.2",
    "pt_1p2_1p8": "1.2 < p_{T} < 1.8",
    "pt_1p8_5p0": "1.8 < p_{T} < 5.0",
}

# Available methods:
#   weighted_peak, global_max, projection_max, weighted_median, centroid, global_peak_region
selected_method = "centroid"
weighted_peak_radius_bins = 80
fraction_of_max_set = 0.05

minimum_hist_integral = 0.5
minimum_categories = 2
require_both_charges = True

# Example detector location.
example_side = "side0"
example_sector = 3
example_phi_bin = 1
example_layer_bin = 2

# Requested module convention:
# R1 module = sector
# R2 module = 12 + sector
# R3 module = 24 + sector
example_modules = {
    "R1": example_sector,
    "R2": 12 + example_sector,
    "R3": 24 + example_sector,
}

save_pdf = False
save_png = False
write_root_output = False


## Vote-map orientation

The vote histograms use

\[
x=\Delta r,\qquad y=r\Delta\phi.
\]

For high-\(p_T\) tracks, \(r\Delta\phi\) should be better constrained while
\(\Delta r\) remains weakly constrained. Therefore the high-\(p_T\) vote
structure should be approximately **horizontal**, not vertical.


In [3]:
# ============================================================
# ROOT and plotting helpers
# ============================================================

root_file = root.TFile.Open(str(input_file), "READ")
if not root_file or root_file.IsZombie():
    raise OSError(f"Could not open {input_file}")

keep_objects = []
canvases = []

def keep(obj):
    keep_objects.append(obj)
    return obj

def vote_path(side, charge, pt_bin, module, phi_bin, layer_bin):
    return (
        f"{selection_name}/{side}/{charge}/{pt_bin}/module_{module:02d}/"
        f"phi_{phi_bin}_layer_{layer_bin}/"
        "h_deltaRPhi_vs_deltaR_votes"
    )

def get_vote_hist(side, charge, pt_bin, module, phi_bin, layer_bin):
    return root_file.Get(
        vote_path(
            side,
            charge,
            pt_bin,
            module,
            phi_bin,
            layer_bin,
        )
    )

def clone_normalized(hist, name):
    if not hist:
        return None

    integral = hist.Integral()
    if not math.isfinite(integral) or integral < minimum_hist_integral:
        return None

    out = hist.Clone(name)
    out.SetDirectory(0)
    out.Scale(1.0 / integral)
    return keep(out)

def set_pad(logz=False):
    root.gPad.SetLeftMargin(0.13)
    root.gPad.SetBottomMargin(0.13)
    root.gPad.SetRightMargin(0.16)
    root.gPad.SetTopMargin(0.07)
    root.gPad.SetLogz(logz)

def draw_label(text, x=0.15, y=0.94, size=0.030):
    label = keep(root.TLatex())
    label.SetNDC(True)
    label.SetTextFont(42)
    label.SetTextSize(size)
    label.DrawLatex(x, y, text)
    return label

def save_canvas(canvas, name):
    canvases.append(canvas)

    if save_pdf:
        canvas.SaveAs(str(plot_dir / f"{name}.pdf"))

    if save_png:
        canvas.SaveAs(str(plot_dir / f"{name}.png"))


In [4]:
# ============================================================
# 2D correction estimators
# ============================================================

def global_max_2d(hist):
    if not hist or hist.Integral() <= 0:
        return None

    best = None

    for ix in range(1, hist.GetNbinsX() + 1):
        for iy in range(1, hist.GetNbinsY() + 1):
            value = hist.GetBinContent(ix, iy)

            if best is None or value > best["weight"]:
                best = {
                    "dr": hist.GetXaxis().GetBinCenter(ix),
                    "drphi": hist.GetYaxis().GetBinCenter(iy),
                    "weight": value,
                    "ix": ix,
                    "iy": iy,
                }

    if best is None or best["weight"] <= 0:
        return None

    return best

def weighted_peak_2d(hist, radius_bins=2):
    maximum = global_max_2d(hist)
    if maximum is None:
        return None

    ix_min = max(1, maximum["ix"] - radius_bins)
    ix_max = min(hist.GetNbinsX(), maximum["ix"] + radius_bins)
    iy_min = max(1, maximum["iy"] - radius_bins)
    iy_max = min(hist.GetNbinsY(), maximum["iy"] + radius_bins)

    sum_weight = 0.0
    sum_dr = 0.0
    sum_drphi = 0.0

    for ix in range(ix_min, ix_max + 1):
        for iy in range(iy_min, iy_max + 1):
            weight = hist.GetBinContent(ix, iy)
            if weight <= 0:
                continue

            sum_weight += weight
            sum_dr += weight * hist.GetXaxis().GetBinCenter(ix)
            sum_drphi += weight * hist.GetYaxis().GetBinCenter(iy)

    if sum_weight <= 0:
        return None

    return {
        "dr": sum_dr / sum_weight,
        "drphi": sum_drphi / sum_weight,
        "weight": maximum["weight"],
    }

def global_peak_region_2d(
    hist,
    fraction_of_max=fraction_of_max_set,
    minimum_bins=3,
):
    """
    Find the centroid of the global high-density peak region.

    1. Find the global maximum bin.
    2. Keep bins above fraction_of_max * maximum.
    3. Retain only the connected region containing the global maximum.
    4. Return its weighted centroid.
    """

    if not hist or hist.Integral() <= 0:
        return None

    nx = hist.GetNbinsX()
    ny = hist.GetNbinsY()

    maximum = hist.GetMaximum()
    if maximum <= 0:
        return None

    threshold = fraction_of_max * maximum

    # Locate global maximum.
    max_ix = None
    max_iy = None

    for ix in range(1, nx + 1):
        for iy in range(1, ny + 1):
            if hist.GetBinContent(ix, iy) == maximum:
                max_ix = ix
                max_iy = iy
                break

        if max_ix is not None:
            break

    if max_ix is None:
        return None

    # Flood-fill the connected high-density region.
    stack = [(max_ix, max_iy)]
    visited = set()
    selected_bins = []

    while stack:
        ix, iy = stack.pop()

        if (ix, iy) in visited:
            continue

        visited.add((ix, iy))

        if ix < 1 or ix > nx or iy < 1 or iy > ny:
            continue

        weight = hist.GetBinContent(ix, iy)

        if weight < threshold:
            continue

        selected_bins.append((ix, iy, weight))

        for dx, dy in [
            (-1, 0),
            (1, 0),
            (0, -1),
            (0, 1),
            (-1, -1),
            (-1, 1),
            (1, -1),
            (1, 1),
        ]:
            stack.append((ix + dx, iy + dy))

    if len(selected_bins) < minimum_bins:
        return global_max_2d(hist)

    sum_weight = 0.0
    sum_dr = 0.0
    sum_drphi = 0.0

    for ix, iy, weight in selected_bins:
        dr = hist.GetXaxis().GetBinCenter(ix)
        drphi = hist.GetYaxis().GetBinCenter(iy)

        sum_weight += weight
        sum_dr += weight * dr
        sum_drphi += weight * drphi

    if sum_weight <= 0:
        return None

    return {
        "dr": sum_dr / sum_weight,
        "drphi": sum_drphi / sum_weight,
        "weight": maximum,
        "threshold": threshold,
        "number_of_bins": len(selected_bins),
    }

def projection_max_2d(hist):
    if not hist or hist.Integral() <= 0:
        return None

    hx = hist.ProjectionX(f"{hist.GetName()}_px_{len(keep_objects)}")
    hy = hist.ProjectionY(f"{hist.GetName()}_py_{len(keep_objects)}")
    hx.SetDirectory(0)
    hy.SetDirectory(0)
    keep(hx)
    keep(hy)

    return {
        "dr": hx.GetXaxis().GetBinCenter(hx.GetMaximumBin()),
        "drphi": hy.GetXaxis().GetBinCenter(hy.GetMaximumBin()),
        "weight": hist.GetMaximum(),
    }

def weighted_median_1d(hist):
    total = hist.Integral()
    if total <= 0:
        return None

    target = 0.5 * total
    cumulative = 0.0

    for ibin in range(1, hist.GetNbinsX() + 1):
        cumulative += hist.GetBinContent(ibin)

        if cumulative >= target:
            return hist.GetXaxis().GetBinCenter(ibin)

    return None

def weighted_median_2d(hist):
    if not hist or hist.Integral() <= 0:
        return None

    hx = hist.ProjectionX(f"{hist.GetName()}_mx_{len(keep_objects)}")
    hy = hist.ProjectionY(f"{hist.GetName()}_my_{len(keep_objects)}")
    hx.SetDirectory(0)
    hy.SetDirectory(0)
    keep(hx)
    keep(hy)

    dr = weighted_median_1d(hx)
    drphi = weighted_median_1d(hy)

    if dr is None or drphi is None:
        return None

    return {
        "dr": dr,
        "drphi": drphi,
        "weight": hist.GetMaximum(),
    }

def centroid_2d(hist):
    if not hist or hist.Integral() <= 0:
        return None

    sum_weight = 0.0
    sum_dr = 0.0
    sum_drphi = 0.0

    for ix in range(1, hist.GetNbinsX() + 1):
        for iy in range(1, hist.GetNbinsY() + 1):
            weight = hist.GetBinContent(ix, iy)
            if weight <= 0:
                continue

            sum_weight += weight
            sum_dr += weight * hist.GetXaxis().GetBinCenter(ix)
            sum_drphi += weight * hist.GetYaxis().GetBinCenter(iy)

    if sum_weight <= 0:
        return None

    return {
        "dr": sum_dr / sum_weight,
        "drphi": sum_drphi / sum_weight,
        "weight": hist.GetMaximum(),
    }

def extract_point(hist, method=selected_method):
    if method == "weighted_peak":
        return weighted_peak_2d(hist, weighted_peak_radius_bins)

    if method == "global_max":
        return global_max_2d(hist)

    if method == "global_peak_region":
        return global_peak_region_2d(hist)

    if method == "projection_max":
        return projection_max_2d(hist)

    if method == "weighted_median":
        return weighted_median_2d(hist)

    if method == "centroid":
        return centroid_2d(hist)

    raise ValueError(f"Unknown method: {method}")


## Constraint QA

These plots must be checked before using the cluster correction maps:

- mass after should be fixed to the PDG \(K^0_S\) mass;
- DIRA_xy after should be fixed to 1;
- pairDCA_xy after should approach 0;
- momentum scales should remain near 1;
- rotations should remain small.


In [5]:
# ============================================================
# Constraint QA
# ============================================================

qa_pairs = [
    ("h_mass_before", "h_mass_after", "K^{0}_{S} mass"),
    ("h_dira_xy_before", "h_dira_xy_after", "DIRA_{xy}"),
    ("h_pair_dca_xy_before", "h_pair_dca_xy_after", "pairDCA_{xy}"),
    ("h_dira_3d_before", "h_dira_3d_after", "DIRA_{3D} QA"),
    ("h_pair_dca_3d_before", "h_pair_dca_3d_after", "pairDCA_{3D} QA"),
]

c_constraint_qa = keep(root.TCanvas(
    "c_constraint_qa",
    "c_constraint_qa",
    1500,
    900,
))
c_constraint_qa.Divide(3, 2)

for ipad, (before_name, after_name, label) in enumerate(qa_pairs, start=1):
    c_constraint_qa.cd(ipad)
    set_pad()

    before = root_file.Get(before_name)
    after = root_file.Get(after_name)

    if before:
        before = before.Clone(f"{before_name}_draw")
        before.SetDirectory(0)
        before.SetLineWidth(3)
        keep(before)
        before.Draw("HIST")

    if after:
        after = after.Clone(f"{after_name}_draw")
        after.SetDirectory(0)
        after.SetLineWidth(3)
        after.SetLineStyle(2)
        keep(after)
        after.Draw("HIST SAME" if before else "HIST")

    legend = keep(root.TLegend(0.58, 0.74, 0.88, 0.89))
    legend.SetBorderSize(0)
    legend.SetFillStyle(0)

    if before:
        legend.AddEntry(before, "before", "l")
    if after:
        legend.AddEntry(after, "after", "l")

    legend.Draw()
    draw_label(label, 0.15, 0.94)

c_constraint_qa.cd(6)
set_pad()

h_scale1 = root_file.Get("h_momentum_scale1")
h_scale2 = root_file.Get("h_momentum_scale2")

if h_scale1:
    h_scale1 = h_scale1.Clone("h_momentum_scale1_draw")
    h_scale1.SetDirectory(0)
    h_scale1.SetLineWidth(3)
    keep(h_scale1)
    h_scale1.Draw("HIST")

if h_scale2:
    h_scale2 = h_scale2.Clone("h_momentum_scale2_draw")
    h_scale2.SetDirectory(0)
    h_scale2.SetLineWidth(3)
    h_scale2.SetLineStyle(2)
    keep(h_scale2)
    h_scale2.Draw("HIST SAME" if h_scale1 else "HIST")

legend = keep(root.TLegend(0.58, 0.74, 0.88, 0.89))
legend.SetBorderSize(0)
legend.SetFillStyle(0)

if h_scale1:
    legend.AddEntry(h_scale1, "scale 1", "l")
if h_scale2:
    legend.AddEntry(h_scale2, "scale 2", "l")

legend.Draw()
draw_label("Momentum scales", 0.15, 0.94)

c_constraint_qa.Update()
save_canvas(c_constraint_qa, "constraint_qa_exact_xy")
c_constraint_qa


## \(R1\), \(R2\), and \(R3\) examples with explicit \(p_T\) bins

The three columns use the requested module convention:

- \(R1:\) `module = sector`;
- \(R2:\) `module = 12 + sector`;
- \(R3:\) `module = 24 + sector`.

The five rows retain the separate daughter-\(p_T\) bins. Positive and negative
charges are independently normalized before they are averaged.


In [6]:
# ============================================================
# pT-binned R1/R2/R3 examples
# ============================================================

c_pt_examples = keep(root.TCanvas(
    "c_pt_examples",
    "c_pt_examples",
    1500,
    1800,
))
c_pt_examples.Divide(3, len(pt_bins))

for column, (region, module) in enumerate(example_modules.items()):
    for row, pt_bin in enumerate(pt_bins):
        ipad = row * 3 + column + 1
        c_pt_examples.cd(ipad)
        set_pad()

        combined = None
        used = 0

        for charge in charges:
            source = get_vote_hist(
                example_side,
                charge,
                pt_bin,
                module,
                example_phi_bin,
                example_layer_bin,
            )

            normalized = clone_normalized(
                source,
                (
                    f"h_example_{region}_{pt_bin}_{charge}_"
                    f"{example_phi_bin}_{example_layer_bin}"
                ),
            )

            if normalized is None:
                continue

            if combined is None:
                combined = normalized.Clone(
                    f"h_example_combined_{region}_{pt_bin}"
                )
                combined.SetDirectory(0)
                keep(combined)
            else:
                combined.Add(normalized)

            used += 1

        if combined is None:
            draw_label("missing", 0.34, 0.50, 0.050)
            continue

        combined.Scale(1.0 / used)
        combined.Draw("COLZ")

        point = weighted_peak_2d(
            combined,
            weighted_peak_radius_bins,
        )

        if point is not None:
            marker = keep(root.TMarker(
                point["dr"],
                point["drphi"],
                20,
            ))
            marker.SetMarkerSize(1.7)
            marker.Draw("SAME")

        draw_label(
            f"{region}, module {module:02d}",
            0.14,
            0.94,
            0.028,
        )
        draw_label(
            f"{pt_labels[pt_bin]} GeV/c",
            0.14,
            0.89,
            0.026,
        )

c_pt_examples.Update()
save_canvas(c_pt_examples, "pt_binned_R1_R2_R3_examples")
c_pt_examples


## What the \(R1/R2/R3\) panel shows

The preceding \(5\times3\) canvas keeps the daughter-\(p_T\) bins **separate**.

- Each **row** is one \(p_T\) interval.
- Each **column** is one radial TPC region: \(R1\), \(R2\), or \(R3\).
- Inside each panel, the positive- and negative-charge histograms are normalized independently and then averaged.
- No different \(p_T\) intervals are mixed in this canvas.

Therefore each panel answers:

> What correction-line pattern is produced by one \(p_T\) interval in one radial region after averaging the two charge signs?

This is the cleanest illustration of how the vote-line orientation changes with track curvature.


## Cumulative charge-\(p_T\) construction

The next section shows a different view.

Starting from an empty histogram, normalized source histograms are added one at a time in this order:

1. first \(p_T\) interval, positive charge;
2. first \(p_T\) interval, negative charge;
3. second \(p_T\) interval, positive charge;
4. second \(p_T\) interval, negative charge;
5. and so on.

After every addition, the cumulative histogram is renormalized for display.

This sequence illustrates how the initially broad correction lines from individual charge-\(p_T\) categories gradually intersect and produce a localized common maximum.

The cumulative plots are **diagnostic illustrations only**. The final correction-map construction later in the notebook combines all valid charge-\(p_T\) categories with equal total weight in each detector bin.


In [7]:
# ============================================================
# Cumulative charge-pT illustration for one detector bin
# ============================================================

selected_side = example_side
selected_module = example_modules["R2"]
selected_phi_bin = example_phi_bin
selected_layer_bin = example_layer_bin

draw_peak = True

cumulative_objects = []
cumulative_canvases = []
cumulative = None
number_added = 0

for pt_bin in pt_bins:
    for charge in charges:
        source = get_vote_hist(
            selected_side,
            charge,
            pt_bin,
            selected_module,
            selected_phi_bin,
            selected_layer_bin,
        )

        normalized = clone_normalized(
            source,
            f"h_cumulative_source_{charge}_{pt_bin}",
        )

        if normalized is None:
            print(f"Missing: {charge}, {pt_bin}")
            continue

        if cumulative is None:
            cumulative = normalized.Clone(
                "h_cumulative_charge_pt"
            )
            cumulative.SetDirectory(0)
            keep(cumulative)
        else:
            cumulative.Add(normalized)

        number_added += 1

        shown = cumulative.Clone(
            f"h_cumulative_shown_{charge}_{pt_bin}"
        )
        shown.SetDirectory(0)
        shown.Scale(1.0 / number_added)
        keep(shown)

        shown.SetTitle(
            (
                f"{selected_side}, module {selected_module:02d}, "
                f"#phi bin {selected_phi_bin}, "
                f"layer bin {selected_layer_bin};"
                "#Delta r [cm];r#Delta#phi [cm]"
            )
        )

        canvas = keep(root.TCanvas(
            f"c_cumulative_{charge}_{pt_bin}",
            f"c_cumulative_{charge}_{pt_bin}",
            800,
            650,
        ))
        canvas.SetLeftMargin(0.12)
        canvas.SetBottomMargin(0.12)
        canvas.SetRightMargin(0.15)
        canvas.SetTopMargin(0.08)

        shown.Draw("COLZ")

        peak = weighted_peak_2d(
            shown,
            weighted_peak_radius_bins,
        )

        marker = None
        if draw_peak and peak is not None:
            marker = keep(root.TMarker(
                peak["dr"],
                peak["drphi"],
                20,
            ))
            marker.SetMarkerSize(1.3)
            marker.Draw("SAME")

        label = keep(root.TLatex())
        label.SetNDC(True)
        label.SetTextFont(42)
        label.SetTextSize(0.034)
        label.DrawLatex(
            0.13,
            0.94,
            (
                f"Added through {pt_labels[pt_bin]} GeV/c, "
                f"{charge}; N_{{cat}}={number_added}"
            ),
        )

        cumulative_objects.append(
            {
                "canvas": canvas,
                "histogram": shown,
                "marker": marker,
                "label": label,
                "pt_bin": pt_bin,
                "charge": charge,
                "number_added": number_added,
            }
        )

        cumulative_canvases.append(canvas)
        canvas.Draw()


## Compact cumulative summary canvas

The individual canvases above are convenient for stepping through the construction interactively. The next cell collects the same sequence into one multi-panel canvas.

Each panel contains the average of all normalized charge-\(p_T\) categories added up to that stage. The black marker is the local weighted peak used for the correction estimate.


In [8]:
# ============================================================
# Compact cumulative summary
# ============================================================

n_panels = len(cumulative_objects)
n_columns = 5
n_rows = math.ceil(n_panels / n_columns)

c_cumulative_summary = keep(root.TCanvas(
    "c_cumulative_summary",
    "c_cumulative_summary",
    1700,
    330 * n_rows,
))
c_cumulative_summary.Divide(n_columns, n_rows)

for index, item in enumerate(cumulative_objects, start=1):
    c_cumulative_summary.cd(index)
    set_pad()

    histogram = item["histogram"]
    histogram.Draw("COLZ")

    peak = weighted_peak_2d(
        histogram,
        weighted_peak_radius_bins,
    )

    if draw_peak and peak is not None:
        marker = keep(root.TMarker(
            peak["dr"],
            peak["drphi"],
            20,
        ))
        marker.SetMarkerSize(1.1)
        marker.Draw("SAME")

    draw_label(
        (
            f"N_{{cat}}={item['number_added']}: "
            f"{pt_labels[item['pt_bin']]} GeV/c, "
            f"{item['charge']}"
        ),
        0.12,
        0.94,
        0.024,
    )

c_cumulative_summary.Update()
save_canvas(
    c_cumulative_summary,
    "cumulative_charge_pt_construction",
)
c_cumulative_summary


## Interpretation of the two demonstrations

The notebook now contains both complementary visualizations:

### Separate-\(p_T\) view

This view keeps each \(p_T\) interval isolated and averages only the two charge signs. It is used to inspect the geometry of the vote lines:

- high-\(p_T\) daughters should primarily constrain \(r\Delta\phi\);
- lower-\(p_T\) daughters provide stronger sensitivity to \(\Delta r\);
- the line orientation should therefore change with \(p_T\).

### Cumulative view

This view successively overlays normalized charge-\(p_T\) categories. It is used to show how the final two-dimensional correction becomes localized:

- one category may give a broad line or ridge;
- the opposite charge changes the curvature response;
- additional \(p_T\) intervals add differently oriented constraints;
- the common intersection becomes the detector correction estimate.

Neither display changes the final weighting prescription. In the map-building section, every valid charge-\(p_T\) histogram is normalized to unit integral and all valid categories are averaged with equal weight.


## Equal-weight charge-\(p_T\) combination

Every valid source histogram contributes the same total weight:

```python
normalized_histogram = source_histogram / source_histogram.Integral()
combined += normalized_histogram
combined /= number_of_used_categories
```

This weighting is applied independently in every detector bin.


The separate-\(p_T\) and cumulative figures above are visual demonstrations of the same ingredients. The final maps below are always constructed directly from all normalized source categories, not from the displayed cumulative canvases.


In [9]:
# ============================================================
# Build equal-weight combined vote histograms
# ============================================================

combined_hists = {}
used_category_counts = {}
used_charge_counts = {}

all_methods = [
    "weighted_peak",
    "global_max",
    "projection_max",
    "weighted_median",
    "centroid",
]

points_by_method = {
    method: {}
    for method in all_methods
}

for side in sides:
    for module in range(36):
        for phi_bin in range(3):
            for layer_bin in range(5):
                key = (
                    side,
                    module,
                    phi_bin,
                    layer_bin,
                )

                combined = None
                used_categories = 0
                used_charges = set()

                for pt_bin in pt_bins:
                    for charge in charges:
                        source = get_vote_hist(
                            side,
                            charge,
                            pt_bin,
                            module,
                            phi_bin,
                            layer_bin,
                        )

                        normalized = clone_normalized(
                            source,
                            (
                                f"h_norm_{side}_{charge}_{pt_bin}_"
                                f"{module}_{phi_bin}_{layer_bin}"
                            ),
                        )

                        if normalized is None:
                            continue

                        if combined is None:
                            combined = normalized.Clone(
                                (
                                    f"h_combined_{side}_{module}_"
                                    f"{phi_bin}_{layer_bin}"
                                )
                            )
                            combined.SetDirectory(0)
                            keep(combined)
                        else:
                            combined.Add(normalized)

                        used_categories += 1
                        used_charges.add(charge)

                used_category_counts[key] = used_categories
                used_charge_counts[key] = len(used_charges)

                valid = (
                    combined is not None
                    and used_categories >= minimum_categories
                    and (
                        not require_both_charges
                        or len(used_charges) == 2
                    )
                )

                if not valid:
                    combined_hists[key] = None

                    for method in all_methods:
                        points_by_method[method][key] = None

                    continue

                combined.Scale(1.0 / used_categories)
                combined_hists[key] = combined

                for method in all_methods:
                    points_by_method[method][key] = extract_point(
                        combined,
                        method,
                    )

selected_points = points_by_method[selected_method]

print(
    "Valid selected correction bins:",
    sum(point is not None for point in selected_points.values()),
    "/",
    len(selected_points),
)


Valid selected correction bins: 1060 / 1080


In [10]:
# ============================================================
# Make local phi x local layer maps for all modules
# ============================================================

def make_local_map(name, title):
    hist = root.TH2D(
        name,
        title,
        3,
        -0.5,
        2.5,
        5,
        -0.5,
        4.5,
    )
    hist.SetDirectory(0)
    hist.GetXaxis().SetTitle("local #phi bin")
    hist.GetYaxis().SetTitle("local layer bin")
    return keep(hist)

maps = {}

for method in all_methods:
    maps[method] = {}

    for side in sides:
        for module in range(36):
            h_dr = make_local_map(
                f"h_delta_r_{method}_{side}_module_{module:02d}",
                (
                    f"{side}, module {module:02d};"
                    "local #phi bin;local layer bin"
                ),
            )

            h_drphi = make_local_map(
                f"h_delta_rphi_{method}_{side}_module_{module:02d}",
                (
                    f"{side}, module {module:02d};"
                    "local #phi bin;local layer bin"
                ),
            )

            h_categories = make_local_map(
                f"h_used_categories_{side}_module_{module:02d}",
                (
                    f"{side}, module {module:02d};"
                    "local #phi bin;local layer bin"
                ),
            )

            h_valid = make_local_map(
                f"h_valid_{method}_{side}_module_{module:02d}",
                (
                    f"{side}, module {module:02d};"
                    "local #phi bin;local layer bin"
                ),
            )

            for phi_bin in range(3):
                for layer_bin in range(5):
                    key = (
                        side,
                        module,
                        phi_bin,
                        layer_bin,
                    )

                    bin_x = phi_bin + 1
                    bin_y = layer_bin + 1

                    h_categories.SetBinContent(
                        bin_x,
                        bin_y,
                        used_category_counts[key],
                    )

                    point = points_by_method[method][key]
                    if point is None:
                        continue

                    h_dr.SetBinContent(
                        bin_x,
                        bin_y,
                        point["dr"],
                    )

                    h_drphi.SetBinContent(
                        bin_x,
                        bin_y,
                        point["drphi"],
                    )

                    h_valid.SetBinContent(
                        bin_x,
                        bin_y,
                        1.0,
                    )

            maps[method][(side, module)] = {
                "delta_r": h_dr,
                "delta_rphi": h_drphi,
                "used_categories": h_categories,
                "valid": h_valid,
            }

print("Maps created")


Maps created


In [11]:
# ============================================================
# Show selected-method maps for the example R1/R2/R3 modules
# ============================================================

c_final_examples = keep(root.TCanvas(
    "c_final_examples",
    "c_final_examples",
    1500,
    900,
))
c_final_examples.Divide(3, 2)

for column, (region, module) in enumerate(example_modules.items()):
    c_final_examples.cd(column + 1)
    set_pad()

    maps[selected_method][
        (example_side, module)
    ]["delta_r"].Draw("COLZ TEXT")

    draw_label(
        f"{region}: #Delta r",
        0.15,
        0.94,
    )

    c_final_examples.cd(column + 4)
    set_pad()

    maps[selected_method][
        (example_side, module)
    ]["delta_rphi"].Draw("COLZ TEXT")

    draw_label(
        f"{region}: r#Delta#phi",
        0.15,
        0.94,
    )

c_final_examples.Update()
save_canvas(c_final_examples, "final_R1_R2_R3_maps")
c_final_examples


In [12]:
# ============================================================
# Compare peak definitions for one example module
# ============================================================

comparison_methods = [
    "global_max",
    "weighted_peak",
    "weighted_median",
]

comparison_module = example_modules["R2"]

c_method_comparison = keep(root.TCanvas(
    "c_method_comparison",
    "c_method_comparison",
    1500,
    900,
))
c_method_comparison.Divide(3, 2)

for column, method in enumerate(comparison_methods):
    c_method_comparison.cd(column + 1)
    set_pad()

    maps[method][
        (example_side, comparison_module)
    ]["delta_r"].Draw("COLZ TEXT")

    draw_label(f"#Delta r: {method}", 0.15, 0.94)

    c_method_comparison.cd(column + 4)
    set_pad()

    maps[method][
        (example_side, comparison_module)
    ]["delta_rphi"].Draw("COLZ TEXT")

    draw_label(f"r#Delta#phi: {method}", 0.15, 0.94)

c_method_comparison.Update()
save_canvas(c_method_comparison, "peak_method_comparison")
c_method_comparison


In [13]:
# ============================================================
# Write correction ROOT file
# ============================================================

if write_root_output:
    output_file.parent.mkdir(parents=True, exist_ok=True)

    output = root.TFile.Open(
        str(output_file),
        "RECREATE",
    )

    if not output or output.IsZombie():
        raise OSError(f"Could not create {output_file}")

    selected_directory = output.mkdir("selected")
    selected_directory.cd()

    root.TNamed(
        "selected_method",
        selected_method,
    ).Write()

    root.TNamed(
        "weighting",
        (
            "Each available charge-pT vote histogram is normalized "
            "to unit integral before equal-weight averaging."
        ),
    ).Write()

    root.TNamed(
        "module_numbering",
        (
            "0-11 R1 sectors 0-11; "
            "12-23 R2 sectors 0-11; "
            "24-35 R3 sectors 0-11"
        ),
    ).Write()

    for (side, module), module_maps in maps[selected_method].items():
        side_directory = (
            selected_directory.GetDirectory(side)
            or selected_directory.mkdir(side)
        )

        module_directory = side_directory.mkdir(
            f"module_{module:02d}"
        )
        module_directory.cd()

        for hist in module_maps.values():
            hist.Write()

    diagnostics_directory = output.mkdir(
        "method_diagnostics"
    )

    for method in all_methods:
        method_directory = diagnostics_directory.mkdir(
            method
        )

        for (side, module), module_maps in maps[method].items():
            side_directory = (
                method_directory.GetDirectory(side)
                or method_directory.mkdir(side)
            )

            module_directory = side_directory.mkdir(
                f"module_{module:02d}"
            )
            module_directory.cd()

            module_maps["delta_r"].Write()
            module_maps["delta_rphi"].Write()
            module_maps["valid"].Write()

    output.Close()
    print(f"Wrote {output_file}")


In [14]:
# ============================================================
# Optional multipage PDF
# ============================================================

if save_pdf and canvases:
    summary_pdf = str(
        plot_dir / "k0s_2d_correction_summary.pdf"
    )

    canvases[0].Print(summary_pdf + "[")

    for canvas in canvases:
        canvas.Print(summary_pdf)

    canvases[-1].Print(summary_pdf + "]")

    print(f"Wrote {summary_pdf}")
